In [11]:
# ==============================
# 1. Import Libraries
# ==============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression 

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample


# ==============================
# 2. Load Dataset
# ==============================
dataset = pd.read_csv(
    'dataset/email_spam.csv',
    header=None,
    encoding='latin1',
    names=['Status', 'Message'],
    usecols=['Status', 'Message']
)

# ==============================
# 3. Data Cleaning
# ==============================
data = dataset.dropna()

data = data[data['Status'].isin(['ham', 'spam'])]
data['Status'] = data['Status'].replace({'ham': 1, 'spam': 0})

# ==============================
# 🔥 4. Balance Data
# ==============================
spam = data[data.Status == 0]
ham  = data[data.Status == 1]

spam_upsampled = resample(
    spam,
    replace=True,
    n_samples=len(ham),
    random_state=42
)

data = pd.concat([ham, spam_upsampled])

# ==============================
# 5. Split Data
# ==============================
X = data['Message']
y = data['Status'].astype(int)

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=4
)

# ==============================
# 6. Vectorization
# ==============================
vectorizer = TfidfVectorizer(
    min_df=2,
    ngram_range=(1,2),
    stop_words='english'
)

x_traincv = vectorizer.fit_transform(x_train)
x_testcv = vectorizer.transform(x_test)

# ==============================
# 7. Train Models
# ==============================
modelNB = MultinomialNB()

modelLR = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

modelNB.fit(x_traincv, y_train)
modelLR.fit(x_traincv, y_train)

# ==============================
# 8. Predictions
# ==============================
y_pred_nb = modelNB.predict(x_testcv)
y_pred_lr = modelLR.predict(x_testcv)

# ==============================
# 9. Evaluation NB
# ==============================
print("===== Naive Bayes =====")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

# ==============================
# 10. Evaluation LR
# ==============================
print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

# ==============================
# 11. Test Custom Message
# ==============================
sample = ["Your order has been shipped and will arrive in 2 days"]

sample_vec = vectorizer.transform(sample)

predictionNB = modelNB.predict(sample_vec)
predictionLR = modelLR.predict(sample_vec)

print("\nNB Prediction:", "Spam" if predictionNB[0] == 0 else "Ham")
print("LR Prediction:", "Spam" if predictionLR[0] == 0 else "Ham")

===== Naive Bayes =====
Accuracy: 0.9761658031088083
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       957
           1       0.99      0.96      0.98       973

    accuracy                           0.98      1930
   macro avg       0.98      0.98      0.98      1930
weighted avg       0.98      0.98      0.98      1930

[[946  11]
 [ 35 938]]

===== Logistic Regression =====
Accuracy: 0.9953367875647668
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       957
           1       1.00      0.99      1.00       973

    accuracy                           1.00      1930
   macro avg       1.00      1.00      1.00      1930
weighted avg       1.00      1.00      1.00      1930

[[955   2]
 [  7 966]]

NB Prediction: Spam
LR Prediction: Ham
